In [2]:
# ──End-to-End Demo ──
# OCPP IDS: Load → Infer → Detect Drift → Query Human → Update Model
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import time
import os
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("OCPP IDS — End-to-End Demo")
print("Milestones 1–4: Full Pipeline")
print("=" * 60)

# ── MLP architecture ───
class MLP(nn.Module):
    def __init__(self, input_dim, hidden=[128,64,32],
                 num_classes=5, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(),
                       nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

# ── Load data ──
BASE = os.path.dirname(os.path.abspath('__file__'))
train_df = pd.read_csv(
    os.path.join(BASE, '../data/ocpp_app_layer/Combined/Train.csv'))
test_df  = pd.read_csv(
    os.path.join(BASE, '../data/ocpp_app_layer/Combined/Test.csv'))

label_col    = 'label'
feature_cols = [c for c in train_df.columns
                if c != label_col and
                train_df[c].dtype in ['float64','int64',
                                       'float32','int32']]
le      = LabelEncoder()
y_train = le.fit_transform(train_df[label_col])
y_test  = le.transform(test_df[label_col])
scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols].values)
X_test  = scaler.transform(test_df[feature_cols].values)

print(f"\n Data loaded")
print(f"   Train: {X_train.shape[0]} samples | "
      f"Test: {X_test.shape[0]} samples | "
      f"Features: {X_train.shape[1]}")
print(f"   Classes: {list(le.classes_)}")

OCPP IDS — End-to-End Demo
Milestones 1–4: Full Pipeline

 Data loaded
   Train: 3020 samples | Test: 1295 samples | Features: 51
   Classes: ['cyberattack_ocpp16_doc_idtag', 'cyberattack_ocpp16_dos_flooding_heartbeat', 'cyberattack_ocpp16_fdi_chargingprofile', 'cyberattack_ocpp16_unauthorized_access', 'normal']


In [4]:
#  — Train Model ──
print("\n" + "="*60)
print("STEP 1: Train MLP with Jitter Augmentation")
print("="*60)

def train_model(X_tr, y_tr, input_dim,
                epochs=50, patience=10, jitter_std=0.05, seed=42):
    torch.manual_seed(seed)
    m         = MLP(input_dim=input_dim)
    optimizer = torch.optim.Adam(m.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    X_t = torch.tensor(X_tr, dtype=torch.float32)
    y_t = torch.tensor(y_tr, dtype=torch.long)
    best_loss, wait, best_state = float('inf'), 0, None
    m.train()
    for epoch in range(epochs):
        idx = torch.randperm(len(X_t))
        epoch_loss = 0
        for i in range(0, len(X_t), 32):
            xb = X_t[idx[i:i+32]] + \
                 torch.randn(len(X_t[idx[i:i+32]]),
                             X_t.shape[1]) * jitter_std
            yb = y_t[idx[i:i+32]]
            optimizer.zero_grad()
            loss = criterion(m(xb), yb)
            loss.backward(); optimizer.step()
            epoch_loss += loss.item()
        if epoch_loss < best_loss:
            best_loss, wait = epoch_loss, 0
            best_state = {k: v.clone()
                          for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    m.load_state_dict(best_state)
    m.eval()
    return m

t0    = time.perf_counter()
model = train_model(X_train, y_train,
                    input_dim=X_train.shape[1])
train_time = time.perf_counter() - t0

with torch.no_grad():
    logits = model(torch.tensor(X_test, dtype=torch.float32))
    preds  = logits.argmax(dim=1).numpy()

acc = accuracy_score(y_test, preds)
f1  = f1_score(y_test, preds, average='macro')
print(f"\n Model trained in {train_time:.2f}s")
print(f"   Accuracy : {acc:.4f}")
print(f"   Macro F1 : {f1:.4f}")

# Save model
os.makedirs('../outputs', exist_ok=True)
torch.save(model.state_dict(), '../outputs/demo_model.pt')
print(f"   Saved  → outputs/demo_model.pt")


STEP 1: Train MLP with Jitter Augmentation

 Model trained in 4.02s
   Accuracy : 0.9992
   Macro F1 : 0.9992
   Saved  → outputs/demo_model.pt


In [6]:
# ── Inference + Confidence Abstention ──
print("\n" + "="*60)
print("STEP 2: Inference with Confidence Abstention")
print("="*60)

ABSTENTION_THRESHOLD = 0.85

def infer(model, X_np, threshold=ABSTENTION_THRESHOLD):
    with torch.no_grad():
        logits = model(torch.tensor(X_np, dtype=torch.float32))
        probs  = torch.softmax(logits, dim=1).numpy()
    preds    = np.argmax(probs, axis=1)
    max_conf = probs.max(axis=1)
    abstained = max_conf < threshold
    return preds, probs, max_conf, abstained

# Run inference on 10 sample flows
sample_idx = np.random.choice(len(X_test), 10, replace=False)
X_sample   = X_test[sample_idx]
y_sample   = y_test[sample_idx]

preds, probs, confs, abstained = infer(model, X_sample)

print(f"\n{'Sample':>8} {'True Label':35} "
      f"{'Predicted':35} {'Conf':>7} {'Status':>10}")
print("-" * 100)
for i in range(10):
    true_lbl = le.classes_[y_sample[i]]
    pred_lbl = le.classes_[preds[i]]
    status   = '⚠ ABSTAIN' if abstained[i] else ' ACCEPT'
    match    = '✓' if preds[i] == y_sample[i] else '✗'
    print(f"{sample_idx[i]:>8} {true_lbl:35} "
          f"{pred_lbl:35} {confs[i]:>7.4f} {status:>10} {match}")

n_abstained = abstained.sum()
print(f"\n   Abstained : {n_abstained}/10 "
      f"({n_abstained/10*100:.0f}%) routed to human review")

# Measure latency
lat = []
x_s = torch.tensor(X_test[:1], dtype=torch.float32)
for _ in range(500):
    t0 = time.perf_counter()
    with torch.no_grad(): model(x_s)
    lat.append((time.perf_counter()-t0)*1000)
print(f"   Latency   : p50={np.percentile(lat,50):.4f}ms  "
      f"p90={np.percentile(lat,90):.4f}ms")


STEP 2: Inference with Confidence Abstention

  Sample True Label                          Predicted                              Conf     Status
----------------------------------------------------------------------------------------------------
     772 normal                              normal                               1.0000     ACCEPT ✓
     337 cyberattack_ocpp16_unauthorized_access cyberattack_ocpp16_unauthorized_access  1.0000     ACCEPT ✓
     814 cyberattack_ocpp16_unauthorized_access cyberattack_ocpp16_unauthorized_access  1.0000     ACCEPT ✓
      40 cyberattack_ocpp16_unauthorized_access cyberattack_ocpp16_unauthorized_access  1.0000     ACCEPT ✓
     768 cyberattack_ocpp16_unauthorized_access cyberattack_ocpp16_unauthorized_access  1.0000     ACCEPT ✓
     507 cyberattack_ocpp16_dos_flooding_heartbeat cyberattack_ocpp16_dos_flooding_heartbeat  1.0000     ACCEPT ✓
     918 cyberattack_ocpp16_doc_idtag        cyberattack_ocpp16_doc_idtag         1.0000     ACCEPT ✓
  

In [7]:
# ── Cell 4: Step 3 — Detect Drift ───
print("STEP 3: Detect Distribution Drift (PSI Monitor)")
print("="*60)

from scipy import stats

def compute_psi(reference, current, bins=10):
    bin_min = min(reference.min(), current.min())
    bin_max = max(reference.max(), current.max())
    if bin_max == bin_min: return 0.0
    bin_edges       = np.linspace(bin_min, bin_max, bins+1)
    ref_hist, _     = np.histogram(reference, bins=bin_edges)
    cur_hist, _     = np.histogram(current,   bins=bin_edges)
    ref_prob        = (ref_hist / len(reference)).clip(1e-6)
    cur_prob        = (cur_hist / len(current)).clip(1e-6)
    ref_prob       /= ref_prob.sum()
    cur_prob       /= cur_prob.sum()
    return float(np.sum((ref_prob - cur_prob) *
                        np.log(ref_prob / cur_prob)))

# Simulate drifted production data
np.random.seed(42)
feat_std  = X_train.std(axis=0)
feat_std  = np.where(feat_std == 0, 1e-6, feat_std)

# Simulate Episode: dos attack becomes dominant
dos_idx   = np.where(y_train == 1)[0][:80]
other_idx = np.random.choice(
    np.where(y_train != 1)[0], 120, replace=False)
combined  = np.concatenate([dos_idx, other_idx])
X_drifted = X_train[combined] + \
            np.random.normal(0, 0.15*feat_std,
                             (len(combined), X_train.shape[1]))
y_drifted = y_train[combined]

# Compute PSI on top 5 variance features
feature_variance = X_train.var(axis=0)
top5             = np.argsort(feature_variance)[-5:][::-1]
psi_vals = [compute_psi(X_train[:,fi], X_drifted[:,fi])
            for fi in top5]
mean_psi = np.mean(psi_vals)

print(f"\n   Monitoring Window: 200 production samples")
print(f"   Mean PSI (top-5 features): {mean_psi:.4f}")

if mean_psi > 0.20:
    status = 'CRITICAL — Trigger retraining'
elif mean_psi > 0.10:
    status = ' WARNING — Schedule review'
else:
    status = 'OK — No action needed'

print(f"   Alert status: {status}")
print(f"\n   Per-feature PSI:")
for fi, psi in zip(top5, psi_vals):
    flag = ' ← ALERT' if psi > 0.20 else \
           (' ← WARN' if psi > 0.10 else '')
    print(f"     {feature_cols[fi][:40]:40s}: {psi:.4f}{flag}")

STEP 3: Detect Distribution Drift (PSI Monitor)

   Monitoring Window: 200 production samples
   Mean PSI (top-5 features): 0.7915
   Alert status: CRITICAL — Trigger retraining

   Per-feature PSI:
     fw_websocket_bytes_per_second           : 0.0126
     flow_total_http_2xx_packets             : 0.0180
     flow_avg_ocpp16_metervalues_wh_diff     : 0.0204
     flow_total_RST_flag                     : 0.0189
     dst_port                                : 3.8876 ← ALERT


In [9]:
# ── Query Human (Active Learning) ──
print("\n" + "="*60)
print("STEP 4: Active Learning Query → Human Annotation")
print("="*60)

N_QUERY = 50

def hybrid_query(model, X_pool, n_query):
    with torch.no_grad():
        probs = torch.softmax(
            model(torch.tensor(X_pool, dtype=torch.float32)),
            dim=1).numpy()
    entropy      = -np.sum(probs * np.log(probs+1e-10), axis=1)
    sorted_p     = np.sort(probs, axis=1)[:, ::-1]
    margin       = sorted_p[:,0] - sorted_p[:,1]
    score        = (np.argsort(np.argsort(-entropy)) +
                    np.argsort(np.argsort(margin)))
    return np.argsort(score)[:n_query]

def simulate_annotation(X_q, y_true, noise=0.05):
    y_ann = y_true.copy()
    n_err = int(noise * len(y_ann))
    for idx in np.random.choice(len(y_ann), n_err, replace=False):
        others = [c for c in range(5) if c != y_ann[idx]]
        y_ann[idx] = np.random.choice(others)
    return y_ann

query_idx  = hybrid_query(model, X_drifted, N_QUERY)
X_queried  = X_drifted[query_idx]
y_true_q   = y_drifted[query_idx]
y_annotated = simulate_annotation(X_queried, y_true_q)

n_errors = (y_annotated != y_true_q).sum()
unique, counts = np.unique(y_annotated, return_counts=True)

print(f"\n Hybrid query selected {N_QUERY} samples "
      f"from {len(X_drifted)}-sample pool")
print(f"   Labelling burden reduction: "
      f"{(1 - N_QUERY/len(X_drifted))*100:.1f}%")
print(f"   Annotation errors (simulated): "
      f"{n_errors}/{N_QUERY} ({n_errors/N_QUERY*100:.0f}%)")
print(f"   Class distribution of queried samples:")
for cls, cnt in zip(le.classes_[unique], counts):
    print(f"     {cls}: {cnt}")


STEP 4: Active Learning Query → Human Annotation

 Hybrid query selected 50 samples from 200-sample pool
   Labelling burden reduction: 75.0%
   Annotation errors (simulated): 2/50 (4%)
   Class distribution of queried samples:
     cyberattack_ocpp16_dos_flooding_heartbeat: 25
     cyberattack_ocpp16_fdi_chargingprofile: 15
     cyberattack_ocpp16_unauthorized_access: 1
     normal: 9


In [ ]:
# ── Update Model (Experience Replay) ──
print("\n" + "="*60)
print("STEP 5: Continual Update with Experience Replay")
print("="*60)

# Replay buffer
class ReplayBuffer:
    def __init__(self, max_size=500, n_classes=5):
        self.per_class = max_size // n_classes
        self.X_buf, self.y_buf = {}, {}
        self.rng = np.random.RandomState(42)
    def add(self, X, y):
        for cls in range(5):
            mask = (y == cls)
            Xc, yc = X[mask], y[mask]
            if len(Xc) == 0: continue
            if cls not in self.X_buf:
                self.X_buf[cls] = Xc[:self.per_class]
                self.y_buf[cls] = yc[:self.per_class]
            else:
                cX = np.vstack([self.X_buf[cls], Xc])
                cy = np.concatenate([self.y_buf[cls], yc])
                if len(cX) > self.per_class:
                    idx = self.rng.choice(
                        len(cX), self.per_class, replace=False)
                    cX, cy = cX[idx], cy[idx]
                self.X_buf[cls] = cX
                self.y_buf[cls] = cy
    def sample(self):
        if not self.X_buf: return None, None
        return (np.vstack(list(self.X_buf.values())),
                np.concatenate(list(self.y_buf.values())))

# Initialise buffer with training data
buf = ReplayBuffer()
buf.add(X_train, y_train)

# Add annotated samples
buf.add(X_queried, y_annotated)
X_buf, y_buf = buf.sample()

# Mix: new annotated + buffer
X_mixed = np.vstack([X_queried, X_buf])
y_mixed = np.concatenate([y_annotated, y_buf])

# Evaluate BEFORE update
with torch.no_grad():
    p_before = logits = model(
        torch.tensor(X_drifted, dtype=torch.float32))
    p_before = torch.softmax(p_before, dim=1).numpy()
    pred_before = np.argmax(p_before, axis=1)
f1_before = f1_score(y_drifted, pred_before, average='macro')

# Retrain
t0 = time.perf_counter()
model_updated = train_model(X_mixed, y_mixed,
                             input_dim=X_train.shape[1],
                             epochs=30, patience=7)
update_time = time.perf_counter() - t0

# Evaluate AFTER update
with torch.no_grad():
    p_after = torch.softmax(
        model_updated(
            torch.tensor(X_drifted, dtype=torch.float32)),
        dim=1).numpy()
    pred_after = np.argmax(p_after, axis=1)
f1_after_drift = f1_score(y_drifted, pred_after, average='macro')

with torch.no_grad():
    p_clean = torch.softmax(
        model_updated(
            torch.tensor(X_test, dtype=torch.float32)),
        dim=1).numpy()
    pred_clean = np.argmax(p_clean, axis=1)
f1_clean = f1_score(y_test, pred_clean, average='macro')

print(f"\n Continual update complete in {update_time:.2f}s")
print(f"   Mixed dataset size  : {len(X_mixed)} samples")
print(f"   F1 on drifted data  : "
      f"{f1_before:.4f} → {f1_after_drift:.4f} "
      f"({'↑' if f1_after_drift > f1_before else '↓'})")
print(f"   F1 on clean test    : {f1_clean:.4f}")
slo_status = ' PASS' if f1_clean >= 0.97 else '❌ FAIL'
print(f"   SLO validation      : {slo_status}")

# Save updated model
torch.save(model_updated.state_dict(),
           '../outputs/demo_model_updated.pt')
print(f"   Saved → outputs/demo_model_updated.pt")


STEP 5: Continual Update with Experience Replay

 Continual update complete in 0.54s
   Mixed dataset size  : 550 samples
   F1 on drifted data  : 1.0000 → 1.0000 (↓)
   F1 on clean test    : 0.9977
   SLO validation      :  PASS
   Saved → outputs/demo_model_updated.pt


In [11]:
# ──  Demo Summary ──
print("\n" + "="*60)
print("DEMO COMPLETE — Full Pipeline Summary")
print("="*60)

print(f"""
  Step 1 — Train       : F1={f1:.4f}  Time={train_time:.2f}s
  Step 2 — Infer       : p50 latency={np.percentile(lat,50):.4f}ms
                         Abstained={n_abstained}/10
  Step 3 — Drift       : Mean PSI={mean_psi:.4f}
                         Status={status}
  Step 4 — AL Query    : {N_QUERY} samples queried
                         Errors={n_errors}/{N_QUERY}
  Step 5 — CL Update   : F1 drifted {f1_before:.4f}→{f1_after_drift:.4f}
                         F1 clean={f1_clean:.4f}
                         Update time={update_time:.2f}s

  Saved artifacts:
    outputs/demo_model.pt
    outputs/demo_model_updated.pt
""")

print("="*60)
print("All outputs saved to outputs/ directory")
print("="*60)


DEMO COMPLETE — Full Pipeline Summary

  Step 1 — Train       : F1=0.9992  Time=4.02s
  Step 2 — Infer       : p50 latency=0.0545ms
                         Abstained=0/10
  Step 3 — Drift       : Mean PSI=0.7915
                         Status=CRITICAL — Trigger retraining
  Step 4 — AL Query    : 50 samples queried
                         Errors=2/50
  Step 5 — CL Update   : F1 drifted 1.0000→1.0000
                         F1 clean=0.9977
                         Update time=0.54s

  Saved artifacts:
    outputs/demo_model.pt
    outputs/demo_model_updated.pt

All outputs saved to outputs/ directory
